it's been too long since I just built something.

so i'm gonna build.

deepseek moe from scratch.

In [5]:
import torch
import numpy

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [ ]:
def SDPA(query, key, value,
                 scaling_factor=None,
                 mask=None):

    """
    Compute Scaled Dot Product Attention

    INPUTS: query, key  : B, N_H, T, HEAD_DIM_QK
            value       : B, N_H, T, HEAD_DIM_V
            scaling_factor : SCALAR,
            mask        : T, T 

    OUTPUTS:
            attention_scores : B, N_H, T, HEAD_DIM_V
    """

    if scaling_factor == None:
        scaling_factor = query.shape[-1] ** -0.5

    affinities = torch.matmul(query, key.transpose(-2, -1)) / scaling_factor

    if mask is not None:
        affinities._masked_fill(mask==0, -1e9)
    else:
        # block_size x block_size
        mask=torch.tril(torch.ones(query.shape[-2], query.shape[-2]))
        affinities._masked_fill(mask==0, -1e9)

    probs = affinities.softmax(dim=-1)

    return torch.matmul(probs, value)
       

NameError: name 'torch' is not defined

In [ ]:
class self_MHA(torch.nn.Module):
    def __init__(self, embd_dim, n_heads, bias=True,):
        super().__init__()

        self.packed_qkv_proj = torch.nn.Linear(embd_dim, embd_dim * 3, bias=bias)

        self.out_proj = torch.nn.Linear(embd_dim, embd_dim)

        assert embd_dim % n_heads == 0, "Embedding dim not divisible by num of heads, make it a multiple!"

        self.head_dim = embd_dim // n_heads
        self.n_heads = n_heads
        
    def forward(self, x):

        qkv = self.packed_qkv_proj(x)

        # q, k, v: B, T, embd_dim
        # goal is to get B, T, n_h, head_dim
        q,k,v = torch.chunk(qkv, chunks=3, dim=-1)

        q = q.unflatten(-1, [self.n_heads, self.head_dim]).transpose(1, 2)
        k = k.unflatten(-1, [self.n_heads, self.head_dim]).transpose(1, 2)
        v = v.unflatten(-1, [self.n_heads, self.head_dim]).transpose(1, 2)

        # flatten on the num_heads to get concatenated output
        attention_scores = SDPA(q, k, v).transpose(1,2).flatten(-2)

        out = self.out_proj(attention_scores)

        return out

In [ ]:
class Decoder(torch.nn.Module):

    # compute self attention and then ffn and thats it?

    def __init__(self):
        super().__init__()

        

NameError: name 'torch' is not defined

In [ ]:
class SimpleTransformer(torch.nn.Module):

    def __init__(self, n_blocks=4):
        super().__init__()

        self.transformer_blocks = torch.nn.TransformerDecoder(torch.nn.TransformerDecoderLayer(), num_layers=n_blocks

    def forward(self, x, y=None):
        pass

    def generate(self, x, max_tokens=100):
        pass